In [6]:
import sys
import os

parent_dir = os.path.abspath(os.path.join(os.getcwd(), '..'))
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)

# necessary tests for Section 3
from tests.test import (
    test_thresholds,
    test_number_of_cells,
    test_cell_boundaries,
    test_real_values_to_cell_indices,
    test_sample_to_cell_indices,
    test_tau_mapping,
    test_threshold_equivalence
)

print("Setup complete. Tests imported successfully.")

Setup complete. Tests imported successfully.


### 3.1 Extracting and Sorting Thresholds

> *"Collect, across all trees of the forest, every threshold that is tested on feature i, and sort the resulting real numbers ℓ0 < ℓ1 < · · · < ℓk−1."*

**Explanation:** The first step in the discretization process is to analyze the entire forest. We look at every single decision node, extract the threshold value used for splitting, group them by feature, and sort them in strictly increasing order. Duplicate thresholds across different trees are merged into a single value.

**Test:** The `test_thresholds()` function verifies that our algorithm correctly scans the sample forest, extracts the thresholds for Feature 0 and Feature 1, removes duplicates, and sorts them.

In [7]:
test_thresholds()

[PASS] Feature 0 thresholds are [2.5, 5.7, 8.1]
[PASS] Feature 1 threshold is [10.0]


### 3.2 Partitioning into Cells

> *"These k thresholds partition the real line into k + 1 intervals, the cells of feature i... Let cells(f) denote the number of cells of feature f induced by the forest."*

**Explanation:** A fundamental property of the real line is that $k$ distinct points (thresholds) divide the line into exactly $k + 1$ segments (cells). For example, if a feature has 3 unique thresholds across all trees, those thresholds will slice the feature's domain into 4 distinct cells.

**Test:** The `test_number_of_cells()` function asserts that for $k$ thresholds, the discretization object correctly reports $k+1$ cells for each respective feature.

In [8]:
test_number_of_cells()

[PASS] Feature 0 has 4 cells and feature 1 has 2 cells


### 3.3 Interval Definitions and Boundaries

> *"intervals: (−∞, ℓ0], (ℓ0, ℓ1], . . . , (ℓk−2, ℓk−1], (ℓk−1, +∞). All intervals are open on the left and closed on the right... This convention follows from the fact that the tests in a random forest are of the form ≤ (less-or-equal)."*

**Explanation:** Standard Decision Trees evaluate conditions as `feature <= threshold`. Because of this strict `<=` operator, any value exactly equal to the threshold goes to the left branch. Therefore, our discretized intervals must be closed on the right side `]`. The real line is thus perfectly covered without overlaps.

**Test:** The `test_cell_boundaries()` function checks if the mathematical boundaries of each calculated interval match this exact definition `(lower_bound, upper_bound]`, handling $-\infty$ and $+\infty$ correctly for the first and last cells.

In [9]:
test_cell_boundaries()

[PASS] cell_bounds(0, 0) is correct
[PASS] cell_bounds(0, 1) is correct
[PASS] cell_bounds(0, 2) is correct
[PASS] cell_bounds(0, 3) is correct


### 3.4 Discretizing Real Values

> *"Each cell receives an integer index 0, 1, . . . , k. A real value of feature i falls in exactly one cell, and the corresponding index is the discretized value of that feature."*

**Explanation:** Once the intervals are defined, any incoming real-valued sample (a float) can be mapped to a specific interval. We simply check which interval the value falls into and assign it the corresponding integer index. This converts continuous data into discrete categories.

**Test:** 
- `test_real_values_to_cell_indices()` tests this logic for single scalar values.
- `test_sample_to_cell_indices()` verifies that a whole array of features (an entire sample row) is correctly translated into an array of integer indices.

In [10]:
test_real_values_to_cell_indices()
test_sample_to_cell_indices()

[PASS] cell_index(0, -1.0) == 0
[PASS] cell_index(0, 2.5) == 0
[PASS] cell_index(0, 4.0) == 1
[PASS] cell_index(0, 5.7) == 1
[PASS] cell_index(0, 7.0) == 2
[PASS] cell_index(0, 8.1) == 2
[PASS] cell_index(0, 9.0) == 3
[PASS] cell_index(1, 9.9) == 0
[PASS] cell_index(1, 10.0) == 0
[PASS] cell_index(1, 10.1) == 1
[PASS] cells_of([7.0, 10.1]) == [2, 1]


### 3.5 Lossless Discretization Property

> *"Proposition 1 (Lossless discretization): The map sending a real sample to its tuple of cell indices is a one-to-one translation under which the forest classifies identically. Any two real values lying in the same cell of a feature are indistinguishable to the forest..."*

**Explanation:** This is the core theorem of the section. Because the forest only branches based on the predefined thresholds, it doesn't matter if a feature value is `5.1` or `5.6` if both fall between the thresholds `5.0` and `5.7`. The tree will route them exactly the same way. By mapping original thresholds to integer thresholds (called `tau`), we can evaluate the tree using only integer math without altering the final prediction.

**Test:** 
- `test_tau_mapping()` verifies that the original float thresholds of internal nodes are correctly mapped to their corresponding integer discrete threshold `tau`.
- `test_threshold_equivalence()` proves the Proposition: it evaluates `value <= threshold` (the original continuous test) and compares it with `cell <= tau` (the discretized test), showing they yield identical boolean results for all ranges of inputs.

In [11]:
test_tau_mapping()
test_threshold_equivalence()

[PASS] tau mapping is correct for every internal node
[PASS] threshold equivalence holds for value -1.0
[PASS] threshold equivalence holds for value 2.5
[PASS] threshold equivalence holds for value 4.0
[PASS] threshold equivalence holds for value 5.7
[PASS] threshold equivalence holds for value 7.0
[PASS] threshold equivalence holds for value 8.1
[PASS] threshold equivalence holds for value 9.0
